# Ella2027 — AMD MI300X + LTX-2.5 (one notebook)

دفتر واحد لإعداد وتشغيل عامل Ella على MI300X. يستخدم صورة AMD الرسمية xDiT ويصل مباشرة إلى طابور Ella الحالي. لا يغيّر Kaggle أو الفرع الرئيسي.

**مهم:** يتطلب MI300X Droplet جاهزاً يعمل عليه Jupyter وDocker وROCm. لا تنفّذ خلية التشغيل إذا لم تتأكد من الرصيد/الفوترة. يتطلب قبول الوصول لموديل LTX-2.5 على Hugging Face.


In [ ]:
import os, pathlib, subprocess, getpass, json, textwrap, time
print('OS:', subprocess.check_output(['bash','-lc','. /etc/os-release; echo $PRETTY_NAME'],text=True).strip())
print('Docker:', subprocess.run(['docker','--version'],capture_output=True,text=True).stdout.strip())
print('GPU nodes:')
subprocess.run(['bash','-lc','ls -l /dev/kfd /dev/dri 2>/dev/null || true'])
assert pathlib.Path('/dev/kfd').exists(), 'AMD /dev/kfd not found. This notebook must run on the MI300X host.'
subprocess.run(['docker','info'],check=True,stdout=subprocess.DEVNULL)
print('Host preflight OK; no GPU container launched yet.')


## Credentials (entered privately; never saved to GitHub)

استخدم نفس Worker Token الموجود في Render باسم `KAGGLE_LTX_WORKER_TOKEN`. توكن Hugging Face يجب أن يكون لحساب قبل الوصول إلى موديل LTX-2.5.


In [ ]:
ELLA_TOKEN = getpass.getpass('Render KAGGLE_LTX_WORKER_TOKEN: ').strip()
HF_TOKEN = getpass.getpass('Hugging Face read token (LTX-2.5 access accepted): ').strip()
if not ELLA_TOKEN: raise ValueError('Ella worker token is required')
if not HF_TOKEN: raise ValueError('Hugging Face token is required')
ROOT = pathlib.Path.home() / 'ella2027-amd-worker'
ROOT.mkdir(parents=True,exist_ok=True)
print('Credentials accepted in memory. They will be passed only to the container, not written to notebook or GitHub.')


## Install and launch the AMD xDiT worker

هذه الخلية تنزّل صورة AMD، وتحمّل موديل LTX-2.5 عند أول تشغيل، ثم تشغّل العامل بشكل دائم. بعد ذلك يلتقط المشاهد تلقائياً من طابور Ella.


In [ ]:
IMAGE='rocm/pytorch-xdit:v26.10'
NAME='ella-amd-ltx25'
WORKER_URL='https://raw.githubusercontent.com/WaelNameh84/Ella2027/feature/amd-mi300x-xdit-ltx25-worker/services/amd-mi300x-worker/amd_worker.py'
worker_path=ROOT/'amd_worker.py'
import urllib.request
urllib.request.urlretrieve(WORKER_URL,worker_path)
subprocess.run(['python3','-m','py_compile',str(worker_path)],check=True)
subprocess.run(['docker','pull',IMAGE],check=True)
existing=subprocess.run(['docker','ps','-a','--format','{{.Names}}'],capture_output=True,text=True,check=True).stdout.splitlines()
if NAME in existing: raise RuntimeError(f'Container {NAME} already exists. Inspect it with docker logs -f {NAME}; refusing to replace it.')
cache=ROOT/'hf-cache'; output=ROOT/'output'; cache.mkdir(exist_ok=True); output.mkdir(exist_ok=True)
cmd=['docker','run','-d','--name',NAME,'--restart','unless-stopped','--cap-add=SYS_PTRACE','--security-opt','seccomp=unconfined','--user','root','--device=/dev/kfd','--device=/dev/dri','--group-add','video','--ipc=host','--network','host','--shm-size','128G','-e','HSA_NO_SCRATCH_RECLAIM=1','-e','OMP_NUM_THREADS=16','-e','CUDA_VISIBLE_DEVICES=0','-e','HF_HOME=/app/huggingface_models','-e',f'HF_TOKEN={HF_TOKEN}','-e',f'ELLA_AMD_WORKER_TOKEN={ELLA_TOKEN}','-e','ELLA_API_BASE_URL=https://ella2027.onrender.com','-v',f'{cache}:/app/huggingface_models','-v',f'{output}:/app/output','-v',f'{worker_path}:/app/amd_worker.py:ro',IMAGE,'bash','-lc','python -m pip install --no-cache-dir requests && xdit --help >/dev/null && apt-get update && apt-get install -y --no-install-recommends ffmpeg && rm -rf /var/lib/apt/lists/* && python /app/amd_worker.py']
print('Launching AMD worker container...')
subprocess.run(cmd,check=True)
print('Container started:',NAME)
print('First startup downloads model weights and initializes the AMD runtime; watch logs in the next cell.')


## Worker status and logs

عرض حالة العامل والسجل. هذه الخلية لا تعيد تشغيل الحاوية.


In [ ]:
print(subprocess.run(['docker','ps','--filter',f'name={NAME}','--format','table {{.Names}}\t{{.Status}}'],capture_output=True,text=True).stdout)
logs=subprocess.run(['docker','logs','--tail','100',NAME],capture_output=True,text=True)
print(logs.stdout); print(logs.stderr)
